# Assignment 3 — Question 2

**Task:** Plot the degree correlations of the real-world network using the $k_{nn}$ vs $k$ plot. Compare the degree correlations with those of the corresponding random graph (averaged over 100 instances).

**Dataset:** Same Email-EU-core network used in Q1 (1005 nodes, 25571 edges).

**Note on data:** The dataset (Email-EU-core) is reused from `Question 1/` automatically. If it is not found there, the notebook will attempt to download it directly. If the download fails, manually download the file from https://snap.stanford.edu/data/email-Eu-core.txt.gz and place it in either the `Question 1/` or `Question 2/` folder.

In [ ]:
import os
import sys
import gzip
import urllib.request
import random
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

random.seed(42)
np.random.seed(42)

# Resolve this notebook's directory so all files are saved alongside it
try:
    SAVE_DIR = os.path.dirname(os.path.abspath(__vsc_ipynb_file__))
except NameError:
    SAVE_DIR = os.path.abspath(os.path.dirname(''))
print(f'Save directory: {SAVE_DIR}')

## 1. Load Real-World Graph

We re-use the Email-EU-core dataset. If the file is already downloaded in Q1, we point to it; otherwise we download it here.

In [ ]:
Q1_DATA    = os.path.join(os.path.dirname(SAVE_DIR), 'Question 1', 'email-Eu-core.txt.gz')
LOCAL_DATA = os.path.join(SAVE_DIR, 'email-Eu-core.txt.gz')
DATA_URL   = 'https://snap.stanford.edu/data/email-Eu-core.txt.gz'

if os.path.exists(Q1_DATA):
    data_path = Q1_DATA
    print(f'Using dataset from Question 1.')
elif os.path.exists(LOCAL_DATA):
    data_path = LOCAL_DATA
    print('Using local dataset.')
else:
    print('Downloading email-Eu-core dataset...')
    urllib.request.urlretrieve(DATA_URL, LOCAL_DATA)
    data_path = LOCAL_DATA
    print('Download complete.')

G_real = nx.Graph()
with gzip.open(data_path, 'rt') as f:
    for line in f:
        line = line.strip()
        if line.startswith('#') or not line:
            continue
        u, v = map(int, line.split())
        if u != v:
            G_real.add_edge(u, v)

print(f'Nodes: {G_real.number_of_nodes()}, Edges: {G_real.number_of_edges()}')

## 2. Compute $k_{nn}(k)$ for the Real Network

The average nearest-neighbour degree $k_{nn}(k)$ is defined as:

$$k_{nn}(k) = \frac{1}{|\{i : k_i = k\}|} \sum_{i: k_i=k} \frac{1}{k_i} \sum_{j \in \mathcal{N}(i)} k_j$$

If $k_{nn}(k)$ increases with $k$ → **assortative** network (high-degree nodes connect to high-degree nodes).  
If $k_{nn}(k)$ decreases with $k$ → **disassortative** network.

In [ ]:
def compute_knn_by_k(G):
    """Compute knn(k): average nearest-neighbour degree binned by degree k."""
    degrees = dict(G.degree())
    knn_by_k = defaultdict(list)

    for node in G.nodes():
        neighbors = list(G.neighbors(node))
        ki = degrees[node]
        if ki == 0:
            continue
        knn_i = np.mean([degrees[n] for n in neighbors])
        knn_by_k[ki].append(knn_i)

    k_sorted = sorted(knn_by_k.keys())
    knn_avg  = [np.mean(knn_by_k[k]) for k in k_sorted]
    return np.array(k_sorted), np.array(knn_avg)


k_real, knn_real = compute_knn_by_k(G_real)
print(f'Degree range: {k_real.min()} – {k_real.max()}')

# Pearson assortativity coefficient as a summary statistic
r = nx.degree_assortativity_coefficient(G_real)
print(f'Pearson assortativity coefficient r = {r:.4f}')

## 3. Configuration Model (Random Graph) — 100 Instances

In [ ]:
def configuration_model(degree_seq):
    """Configuration Model: preserve degree sequence by randomly pairing stubs."""
    stubs = []
    for node, deg in enumerate(degree_seq):
        stubs.extend([node] * deg)
    if len(stubs) % 2 != 0:
        stubs.pop()
    random.shuffle(stubs)

    G = nx.Graph()
    G.add_nodes_from(range(len(degree_seq)))
    for i in range(0, len(stubs) - 1, 2):
        u, v = stubs[i], stubs[i + 1]
        if u != v:
            G.add_edge(u, v)
    return G


N_INSTANCES = 100
orig_degree_seq = [d for _, d in G_real.degree()]

# Collect knn values per degree k across all instances
rand_knn_by_k = defaultdict(list)

print(f'Generating {N_INSTANCES} Configuration Model instances...')
for i in range(N_INSTANCES):
    if (i + 1) % 10 == 0:
        print(f'  Instance {i+1}/{N_INSTANCES}')
    G_rand = configuration_model(orig_degree_seq)
    k_arr, knn_arr = compute_knn_by_k(G_rand)
    for k_val, knn_val in zip(k_arr, knn_arr):
        rand_knn_by_k[k_val].append(knn_val)

print('Done.')

k_rand = sorted(rand_knn_by_k.keys())
knn_rand_mean = np.array([np.mean(rand_knn_by_k[k]) for k in k_rand])
knn_rand_std  = np.array([np.std(rand_knn_by_k[k])  for k in k_rand])

## 4. Plot $k_{nn}$ vs $k$

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

# Real network
ax.scatter(k_real, knn_real, color='steelblue', s=40, zorder=5,
           label=f'Real network (r = {r:.3f})')

# Random (Config. Model) average ± std
ax.plot(k_rand, knn_rand_mean, color='orange', linewidth=2,
        label='Config. Model (avg 100 instances)')
ax.fill_between(k_rand,
                knn_rand_mean - knn_rand_std,
                knn_rand_mean + knn_rand_std,
                color='orange', alpha=0.3, label='±1 std')

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Degree $k$', fontsize=13)
ax.set_ylabel('$k_{nn}(k)$', fontsize=13)
ax.set_title('Degree Correlations: $k_{nn}$ vs $k$\n(Email-EU-core)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, which='both', linestyle='--', alpha=0.4)

plt.tight_layout()
out_path = os.path.join(SAVE_DIR, 'knn_vs_k.png')
plt.savefig(out_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {out_path}')

## 5. Observations

- The Pearson assortativity coefficient **r** summarises the overall correlation: r > 0 → assortative, r < 0 → disassortative, r ≈ 0 → uncorrelated.

- **Real network:** Email networks commonly exhibit slight disassortativity — low-degree peripheral nodes tend to connect to high-degree hub nodes. The $k_{nn}(k)$ curve therefore shows a decreasing trend with $k$.

- **Configuration Model:** By rewiring all edges while preserving only degrees, structural correlations are destroyed. The random-graph baseline is nearly flat (no systematic trend with $k$), as expected for an uncorrelated random graph with the same degree sequence.

- The comparison reveals that the degree correlations observed in the real network are a genuine structural property that goes beyond what can be explained by the degree sequence alone.